# B2 冷链运输温控异常预警算法测试与数据分析（参考答案）

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix


In [2]:
# 读取与清洗数据
df = pd.read_csv('cold_chain_shipments.csv')

# 转换发车时间并提取发车小时
df['depart_time'] = pd.to_datetime(df['depart_time'])
df['depart_hour'] = df['depart_time'].dt.hour

# 按运输单号去重，并删除运输时长或传感器缺失率不合法的记录
df = df.drop_duplicates(subset='shipment_id', keep='first')
df = df[(df['planned_hours'] > 0) & (df['sensor_missing_rate'].between(0, 1))]

# 按货物类型的中位数填补 avg_temp 缺失值
df['avg_temp'] = df['avg_temp'].fillna(df.groupby('cargo_type')['avg_temp'].transform('median'))

print('清洗后记录数：', len(df))


清洗后记录数： 1578


In [3]:
# 统计不同货物类型、不同线路区域的样本数、报警率和平均传感器缺失率
cargo_region_summary = (df.groupby(['cargo_type', 'route_region'])
                        .agg(样本数=('shipment_id', 'count'), 温控报警率=('temp_alarm', 'mean'), 平均传感器缺失率=('sensor_missing_rate', 'mean'))
                        .round(3).reset_index())
print(cargo_region_summary)


   cargo_type route_region  样本数  温控报警率  平均传感器缺失率
0         乳制品           华东   99  0.152     0.080
1         乳制品           华北   78  0.141     0.089
2         乳制品           华南   88  0.318     0.075
3         乳制品           西南   84  0.250     0.082
4        冷冻食品           华东  133  0.090     0.091
5        冷冻食品           华北  111  0.099     0.087
6        冷冻食品           华南  107  0.159     0.072
7        冷冻食品           西南  102  0.176     0.086
8        生鲜食品           华东  167  0.299     0.083
9        生鲜食品           华北  133  0.323     0.083
10       生鲜食品           华南  132  0.462     0.078
11       生鲜食品           西南   98  0.398     0.090
12         疫苗           华东   85  0.412     0.081
13         疫苗           华北   49  0.327     0.076
14         疫苗           华南   70  0.471     0.090
15         疫苗           西南   42  0.429     0.062


In [4]:
feature_cols = [
    'route_region', 'vehicle_type', 'cargo_type', 'planned_hours',
    'outside_temp', 'avg_temp', 'max_temp', 'door_open_count',
    'sensor_missing_rate', 'driver_experience_years', 'depart_hour'
]
category_cols = ['route_region', 'vehicle_type', 'cargo_type']

# 构造特征矩阵与目标变量，并对分类字段进行独热编码
X = pd.get_dummies(df[feature_cols], columns=category_cols)
y = df['temp_alarm']

# 按 75%/25% 划分训练集和测试集，按 y 分层，random_state=42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

# 构建随机森林分类器并完成训练和预测
# 参数：决策树的数量为100，最大深度是6，采用平衡类别权重，random_state=42
model = RandomForestClassifier(n_estimators=100, max_depth=6, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# 计算精确率、召回率、F1值和混淆矩阵
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
print('precision=', round(precision, 3))
print('recall=', round(recall, 3))
print('f1=', round(f1, 3))
print('confusion matrix=\n', cm)

# 计算测试集中不同货物类型的召回率
test_result = df.loc[X_test.index, ['shipment_id', 'cargo_type']].copy()
test_result['actual'] = y_test.to_numpy()
test_result['predicted'] = y_pred
cargo_recall = (test_result[test_result['actual'] == 1].groupby('cargo_type')
                .agg(报警样本数=('actual', 'count'), 召回率=('predicted', 'mean')).round(3))
print(cargo_recall)


precision=

 0.661
recall= 0.673
f1= 0.667
confusion matrix=
 [[251  37]
 [ 35  72]]


            报警样本数    召回率
cargo_type              
乳制品            18  0.611
冷冻食品           15  0.067
生鲜食品           52  0.788
疫苗             22  0.864


In [5]:
# 补充（任务三论据）：分货物类型的漏报数，以及降低阈值对疫苗召回的影响
proba = model.predict_proba(X_test)[:, 1]
test_result['proba'] = proba
for th in (0.5, 0.4, 0.3):
    p = (proba >= th).astype(int)
    print(f'阈值{th}: 总体召回={recall_score(y_test, p):.3f} 精确率={precision_score(y_test, p):.3f}',
          {c: round(g.loc[g.actual == 1, 'proba'].ge(th).mean(), 3) for c, g in test_result.groupby('cargo_type')})
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(8).round(3)
print('\n特征重要性 Top8\n', imp)


阈值0.5: 总体召回=0.673 精确率=0.661 {'乳制品': 0.611, '冷冻食品': 0.067, '生鲜食品': 0.788, '疫苗': 0.864}
阈值0.4: 总体召回=0.738 精确率=0.581 {'乳制品': 0.667, '冷冻食品': 0.333, '生鲜食品': 0.808, '疫苗': 0.909}
阈值0.3: 总体召回=0.869 精确率=0.451 {'乳制品': 0.833, '冷冻食品': 0.6, '生鲜食品': 0.904, '疫苗': 1.0}

特征重要性 Top8
 avg_temp                   0.279
max_temp                   0.271
sensor_missing_rate        0.084
planned_hours              0.071
outside_temp               0.057
door_open_count            0.052
depart_hour                0.036
driver_experience_years    0.033
dtype: float64
